# 패션 플랫폼 설문 데이터 — 전처리 및 ETL

---

## 분석 개요
원본 설문 데이터를 정제하고 MySQL에 적재한다.

| 항목 | 내용 |
|------|------|
| 원본 데이터 | `raw_data/패션_플랫폼_설문_통합.csv` |
| 응답자 수 | 269명 |
| 작업 내용 | 컬럼명 단축, 개인정보 컬럼 제거, 타입 변환, 순서형 변수 처리, Q18 무응답 정리, 논리 모순 검사 |
| 저장 대상 | MySQL `survey` 테이블 |

In [1]:
import pandas as pd
import numpy as np
from pandas.api.types import CategoricalDtype
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
import os

In [2]:
df = pd.read_csv('../raw_data/패션_플랫폼_설문_통합.csv', encoding='utf-8-sig')

In [3]:
print(df.shape)
df.columns.tolist()

(269, 20)


['타임스탬프',
 'Q1. 성별을 선택해주세요.',
 'Q2. 연령대를 선택해주세요.',
 'Q3. 평소 패션/스타일 관련 콘텐츠를 얼마나 자주 찾아보시나요?',
 'Q4. 한 달 평균 패션 관련 지출은 얼마인가요? (의류, 신발, 액세서리 포함)',
 'Q5. 패션 플랫폼을 사용하시나요? (아래 질문에서 "아니오"를 선택하시면 마지막 질문으로 바로 이동합니다.)',
 'Q6. 현재 가장 자주 사용하는 패션 관련 플랫폼을 1~2개 선택해주세요.',
 'Q7. 패션 플랫폼을 선택할 때 가장 중요하게 보는 요소는 무엇인가요? (최대 3개 선택)',
 'Q8. 주로 어떤 목적으로 패션 플랫폼을 열어보나요? (최대 2개 선택)',
 'Q9. 최근 6개월 동안 패션 플랫폼에서 구매한 횟수는 몇 번인가요?',
 'Q10. 패션 플랫폼에서 가장 최근에 구매한 시점은 언제인가요?',
 'Q11. 패션 플랫폼에서 한 번 구매할 때 평균 지출 금액은 얼마인가요?',
 'Q12. 같은 플랫폼에서 다시 구매하게 된 가장 큰 이유는 무엇인가요? (최대 2개 선택)',
 'Q13. 패션 플랫폼에서 구매한 경험이 있다면, 불만족스러웠던 경험이 있나요? (복수 선택)',
 'Q14. 지금 가장 자주 사용하는 플랫폼을 앞으로도 계속 사용할 것 같나요?',
 'Q15. 현재 사용하는 패션 플랫폼을 지인에게 추천할 의향이 얼마나 있나요?',
 'Q16. 지금 사용하는 패션 앱을 처음 알게 된 경로는 무엇인가요?',
 'Q17. 최근 6개월 내 패션 상품 구매에 가장 영향을 준 채널은 무엇인가요?',
 'Q18. (선택) 패션 앱을 사용하면서 아쉬웠던 점이 있다면 자유롭게 적어주세요.',
 '(선택) 추첨 참여를 원하시면 연락 가능한 이메일 주소를 남겨주세요. (당첨 시 기프티콘 발송 용도로만 사용되며, 설문 데이터와 별도로 관리됩니다.)']

---

## 컬럼 단축
긴 질문 텍스트를 분석용 단축 컬럼명으로 변환하고, 분석에 불필요한 이메일 컬럼을 제거한다. 이메일은 추첨 참여 용도이므로 분석 데이터셋에는 포함하지 않는다.

In [4]:
RENAME = {
    '타임스탬프': 'timestamp',
    'Q1. 성별을 선택해주세요.': 'gender',
    'Q2. 연령대를 선택해주세요.': 'age',
    'Q3. 평소 패션/스타일 관련 콘텐츠를 얼마나 자주 찾아보시나요?': 'content_freq',
    'Q4. 한 달 평균 패션 관련 지출은 얼마인가요? (의류, 신발, 액세서리 포함)': 'monthly_spend',
    'Q5. 패션 플랫폼을 사용하시나요? (아래 질문에서 "아니오"를 선택하시면 마지막 질문으로 바로 이동합니다.)': 'uses_platform',
    'Q6. 현재 가장 자주 사용하는 패션 관련 플랫폼을 1~2개 선택해주세요.': 'platforms',
    'Q7. 패션 플랫폼을 선택할 때 가장 중요하게 보는 요소는 무엇인가요? (최대 3개 선택)': 'selection_factors',
    'Q8. 주로 어떤 목적으로 패션 플랫폼을 열어보나요? (최대 2개 선택)': 'open_purpose',
    'Q9. 최근 6개월 동안 패션 플랫폼에서 구매한 횟수는 몇 번인가요?': 'purchase_count',
    'Q10. 패션 플랫폼에서 가장 최근에 구매한 시점은 언제인가요?': 'last_purchase',
    'Q11. 패션 플랫폼에서 한 번 구매할 때 평균 지출 금액은 얼마인가요?': 'avg_spend',
    'Q12. 같은 플랫폼에서 다시 구매하게 된 가장 큰 이유는 무엇인가요? (최대 2개 선택)': 'repurchase_reason',
    'Q13. 패션 플랫폼에서 구매한 경험이 있다면, 불만족스러웠던 경험이 있나요? (복수 선택)': 'dissatisfaction',
    'Q14. 지금 가장 자주 사용하는 플랫폼을 앞으로도 계속 사용할 것 같나요?': 'continue_use',
    'Q15. 현재 사용하는 패션 플랫폼을 지인에게 추천할 의향이 얼마나 있나요?': 'nps',
    'Q16. 지금 사용하는 패션 앱을 처음 알게 된 경로는 무엇인가요?': 'discovery',
    'Q17. 최근 6개월 내 패션 상품 구매에 가장 영향을 준 채널은 무엇인가요?': 'influence',
    'Q18. (선택) 패션 앱을 사용하면서 아쉬웠던 점이 있다면 자유롭게 적어주세요.': 'feedback',
}

df = df.rename(columns=RENAME)
df = df[list(RENAME.values())]

In [5]:
df.columns.tolist()

['timestamp',
 'gender',
 'age',
 'content_freq',
 'monthly_spend',
 'uses_platform',
 'platforms',
 'selection_factors',
 'open_purpose',
 'purchase_count',
 'last_purchase',
 'avg_spend',
 'repurchase_reason',
 'dissatisfaction',
 'continue_use',
 'nps',
 'discovery',
 'influence',
 'feedback']

---

## 타입 변환
timestamp는 datetime으로, nps는 nullable integer로 변환한다.

In [6]:
df['timestamp'] = pd.to_datetime(df['timestamp'])

In [7]:
df['nps'] = df['nps'].astype(pd.Int64Dtype())

---

## Q6 플랫폼명 정규화
옵션 내 콤마 문제를 해결하고, 자유 입력 항목을 표준 플랫폼명으로 통일한다.

In [8]:
# 긴 선택지 원문과 임시 placeholder 정의
# 내부에 쉼표가 있어서 split 전에 미리 치환해둬야 함
LONG_OPTION = '종합 쇼핑몰 (쿠팡, 네이버 쇼핑, 테무, 알리익스프레스 등)'
PLACEHOLDER = '__종합쇼핑몰__'

# 공백으로 붙여 쓴 복합 항목 → 두 개의 플랫폼으로 분리 (1:2 매핑)
SPACE_SPLIT = {
    '자라 룩핀': ['자라', '룩핀'],
}

# 자유 입력 및 표기 변형 → 표준명으로 통일 (1:1 매핑)
PLATFORM_MAP = {
    PLACEHOLDER: '종합 쇼핑몰',   # placeholder → 표준명 복원
    'LOOKPIN': '룩핀',             # 영문 표기 → 한글 통일
    'Shein': '쉬인',               # 영문 표기 → 한글 통일
    'CIDER': '기타',               # 인식 불가 플랫폼 → 기타
    '뉴발 나이키 등': '기타',
    '브랜드 온라인 매장 (사이트)': '기타',
    '에잇세컨즈': '기타',
    '인스타 자사몰': '기타',
    '오까네': '기타',
    '해외어플': '기타',
    '하바티': '기타',
    '아이엠샵': '기타',
    '나인제트': '기타',
    '기타': '기타',
}

def clean_platforms(val):
    # 결측값이면 그대로 반환
    if pd.isna(val):
        return val
    
    # 쉼표 포함된 긴 선택지를 placeholder로 치환 (split 오류 방지)
    s = val.replace(LONG_OPTION, PLACEHOLDER)
    
    # 쉼표 기준으로 분리 후 앞뒤 공백 제거
    items = [x.strip() for x in s.split(',')]
    
    result = []
    for item in items:
        # 공백으로 붙여 쓴 복합 항목이면 두 개로 분리
        if item in SPACE_SPLIT:
            result.extend(SPACE_SPLIT[item])
        # 나머지는 표준명으로 변환 (매핑 없으면 원본 유지)
        else:
            result.append(PLATFORM_MAP.get(item, item))
    
    # 중복 제거 (순서 유지)
    seen = set()
    deduped = [x for x in result if not (x in seen or seen.add(x))]
    
    # 다시 쉼표로 합쳐서 반환
    return ', '.join(deduped)

# 전체 platforms 컬럼에 정제 함수 적용
df['platforms'] = df['platforms'].apply(clean_platforms)

In [9]:
df['platforms'].dropna().sort_values().unique()

array(['29CM', '29CM, W컨셉', '29CM, 종합 쇼핑몰', '4910, 룩핀', '4910, 종합 쇼핑몰',
       'KREAM', 'KREAM, 종합 쇼핑몰', 'KREAM, 지그재그', '기타', '무신사', '무신사, 29CM',
       '무신사, 4910', '무신사, KREAM', '무신사, 기타', '무신사, 에이블리', '무신사, 자라, 룩핀',
       '무신사, 종합 쇼핑몰', '무신사, 지그재그', '무신사, 지그재그, 종합 쇼핑몰', '에이블리',
       '에이블리, 29CM', '에이블리, KREAM', '에이블리, 기타', '에이블리, 종합 쇼핑몰',
       '에이블리, 지그재그', '종합 쇼핑몰', '종합 쇼핑몰, 기타', '종합 쇼핑몰, 쉬인', '지그재그',
       '지그재그, 29CM', '지그재그, 기타', '지그재그, 종합 쇼핑몰'], dtype=object)

---

## Q18 무응답 정리
"없음", "딱히" 등 실질적으로 내용이 없는 응답을 NaN으로 처리한다.

In [10]:
NULL_LIKE = {
    '없음', '없어요', '없습니다', '없어용ㅎㅎ', '없어요.', '없습니다.',
    '없음.', '없어요!', '없어.', '없음 ㅎ', '없어요 ㅎ',
    '-', 'none', '딱히', '딱히.',
    '좋아요', 'nowadays fashion apps are so easy to use',
    '딱히 큰 불편함 없이 사용해옴',
    '한 번도 사용할 적은 없지만 굳이 뽑자면.... 없습니다',
}

df['feedback'] = df['feedback'].apply(
    lambda x: np.nan if pd.notna(x) and str(x).strip().lower() in NULL_LIKE else x
)

---

## Q16 유입 경로 정규화
일부 자유 입력성 응답은 같은 의미라도 표기가 달라 이후 채널 분석에서 분산될 수 있다.
분석 가능한 수준으로만 묶고, 주관적 해석이 큰 항목은 `기타/기억 안남`으로 보수적으로 처리한다.


In [11]:
DISCOVERY_MAP = {
    '앱스토어': '앱스토어/플레이스토어',
    'App store': '앱스토어/플레이스토어',
    '플레이스토어 추천': '앱스토어/플레이스토어',
    '오프라인 광고': '오프라인/미디어 광고',
    '미디어 광고': '오프라인/미디어 광고',
    '유명해서..?': '기타/기억 안남',
    '기억이 안난다': '기타/기억 안남',
}

df['discovery'] = df['discovery'].replace(DISCOVERY_MAP)
df['discovery'].value_counts(dropna=False)


discovery
인스타그램 / 틱톡 / 스레드 등 SNS (광고 및 인플루언서 포함)    84
NaN                                       66
친구 / 지인 추천                                48
포털 검색                                     38
유튜브                                       23
오프라인/미디어 광고                                5
앱스토어/플레이스토어                                3
기타/기억 안남                                   2
Name: count, dtype: int64

### Q16 NaN 해석
Q16의 `NaN`은 Q5에서 `아니오`를 선택한 비사용자가 Q6-Q17을 건너뛰면서 발생한 **구조적 결측**이다.

따라서 유입 경로 분석에서는 플랫폼 사용자만 분모로 사용하며, 해당 `NaN`을 `기타` 같은 별도 응답 범주로 대체하지 않는다.


---

## 순서형 변수 Categorical 변환
분석 시 정렬·비교가 필요한 순서형 변수에 ordered CategoricalDtype을 지정한다.

In [12]:
ORDINAL = {
    'content_freq': CategoricalDtype(
        ['전혀 찾아보지 않는다', '가끔 본다', '보통이다', '자주 본다', '매우 자주 본다'],
        ordered=True
    ),
    'monthly_spend': CategoricalDtype(
        ['5만원 미만', '5~10만원', '10~20만원', '20~30만원', '30만원 이상'],
        ordered=True
    ),
    'purchase_count': CategoricalDtype(
        ['구매하지 않음', '1~2번', '3~5번', '6번 이상'],
        ordered=True
    ),
    'last_purchase': CategoricalDtype(
        ['6개월 이상', '3~6개월', '1~3개월', '1개월 이내'],
        ordered=True
    ),
    'avg_spend': CategoricalDtype(
        ['3만원 미만', '3~7만원', '7~15만원', '15~30만원', '30만원 이상'],
        ordered=True
    ),
    'continue_use': CategoricalDtype(
        ['다른 앱으로 바꿀 것 같다', '아마 사용하지 않을 것 같다', '잘 모르겠다', '아마 사용할 것 같다', '계속 사용할 것 같다'],
        ordered=True
    ),
}

for col, dtype in ORDINAL.items():
    df[col] = df[col].astype(dtype)

---

## 정제 후 다중응답 항목 수 점검
논리 모순으로 제거할 응답과 별개로, 다중응답 문항이 정제 후 몇 개 항목으로 분리되는지 확인한다.

이 표는 Google Forms 제한이 실패했는지 판단하기 위한 것이 아니라, 정제 과정에서 항목 수가 예상보다 늘어난 케이스를 확인하기 위한 참고 점검이다. 특히 Q6의 `종합 쇼핑몰 (쿠팡, 네이버 쇼핑, 테무, 알리익스프레스 등)`처럼 선택지 내부에 쉼표가 있는 문항은 단순 쉼표 분리 시 과대 계산될 수 있으므로, Q6 플랫폼명 정규화 이후의 값을 기준으로 확인한다.

최대 항목 수 초과는 단독 삭제 기준으로 사용하지 않고, 논리 모순 여부는 다음 섹션에서 별도로 판단한다.


In [13]:
def count_multi_select(val):
    if pd.isna(val):
        return 0
    return len([item.strip() for item in str(val).split(', ') if item.strip()])

MAX_CHOICE_RULES = {
    'platforms': 2,
    'selection_factors': 3,
    'open_purpose': 2,
    'repurchase_reason': 2,
}

quality_rows = []
for col, max_allowed in MAX_CHOICE_RULES.items():
    counts = df[col].apply(count_multi_select)
    over_mask = counts > max_allowed
    quality_rows.append({
        '문항': col,
        '기대 최대 항목 수': max_allowed,
        '정제 후 초과 케이스': int(over_mask.sum()),
        '처리': '참고만 하고 삭제하지 않음',
    })

quality_check = pd.DataFrame(quality_rows)
quality_check


,문항,기대 최대 항목 수,정제 후 초과 케이스,처리
0,platforms,2,2,참고만 하고 삭제하지 않음
1,selection_factors,3,1,참고만 하고 삭제하지 않음
2,open_purpose,2,1,참고만 하고 삭제하지 않음
3,repurchase_reason,2,0,참고만 하고 삭제하지 않음


---

## 논리 모순 검사
불성실 응답 제거를 위해 설문 라우팅 구조에서 발생 가능한 모순 케이스를 정의하고 탐지한다.

설문 분기 구조:
- Q5 = '아니오' → Q18로 바로 이동 (Q6-Q17 응답 불가)
- Q5 = '예' + Q9 = '구매하지 않음' → Q10-Q12 스킵, Q13으로 이동
- Q5 = '예' + Q9 ≠ '구매하지 않음' → Q10-Q18 전체 응답

이 기준으로 정의한 6가지 모순 케이스:

| 모순 | 조건 | 비고 |
|------|------|------|
| 모순 1 | Q5 비사용자인데 Q6-Q17 중 응답이 있는 경우 | Google Forms 분기로 사실상 발생 불가, 안전망으로 유지 |
| 모순 2 | Q9 비구매자인데 Q10-Q12 중 응답이 있는 경우 | Google Forms 분기로 사실상 발생 불가, 안전망으로 유지 |
| 모순 3 | Q9 구매자인데 Q10-Q12 중 빠진 응답이 있는 경우 | Google Forms 분기로 사실상 발생 불가, 안전망으로 유지 |
| 모순 4 | Q13 "구매 경험 자체가 없음"인데 Q9에 구매 횟수가 있는 경우 | 실제 탐지 가능 |
| 모순 5 | Q17 "최근 6개월 내 구매 경험 없음"인데 Q9에 구매 횟수가 있는 경우 | 실제 탐지 가능 |
| 모순 6 | Q13 "불만족 경험 없음"과 실제 불만 항목이 함께 선택된 경우 | 실제 탐지 가능 |


In [14]:
# 모순 1: Q5 비사용자인데 Q6~Q17 중 응답이 있는 경우
m1 = (
    (df['uses_platform'] == '아니오') &
    (df['platforms'].notna() | df['nps'].notna() | df['discovery'].notna())
)

# 모순 2: Q9 비구매자인데 Q10~Q12 중 응답이 있는 경우
m2 = (
    (df['purchase_count'] == '구매하지 않음') &
    (df['last_purchase'].notna() | df['avg_spend'].notna() | df['repurchase_reason'].notna())
)

# 모순 3: Q9 구매자인데 Q10~Q12 중 빠진 응답이 있는 경우
m3 = (
    df['purchase_count'].notna() &
    (df['purchase_count'] != '구매하지 않음') &
    (df['last_purchase'].isna() | df['avg_spend'].isna() | df['repurchase_reason'].isna())
)

# 모순 4: Q13 "구매 경험 자체가 없음"인데 Q9에 구매 횟수가 있는 경우
m4 = (
    df['dissatisfaction'].str.contains('구매 경험 자체가 없음', na=False) &
    df['purchase_count'].notna() &
    (df['purchase_count'] != '구매하지 않음')
)

# 모순 5: Q17 "최근 6개월 내 구매 경험 없음"인데 Q9에 구매 횟수가 있는 경우
m5 = (
    (df['influence'] == '최근 6개월 내 패션 상품 구매 경험 없음') &
    df['purchase_count'].notna() &
    (df['purchase_count'] != '구매하지 않음')
)

# 모순 6: Q13 "불만족 경험 없음"과 실제 불만 항목이 함께 선택된 경우
dissatisfaction_items = df['dissatisfaction'].fillna('').str.split(', ')
m6 = dissatisfaction_items.apply(
    lambda items: ('불만족 경험 없음' in items) and (len([x for x in items if x.strip()]) > 1)
)

inconsistent_mask = m1 | m2 | m3 | m4 | m5 | m6
for i, m in enumerate([m1, m2, m3, m4, m5, m6], 1):
    print(f'모순 {i}: {m.sum()}건')
print(f'총 제거: {inconsistent_mask.sum()}건')
df = df[~inconsistent_mask].reset_index(drop=True)

# user_id 부여 (timestamp 정렬 후 1부터)
df = df.sort_values('timestamp').reset_index(drop=True)
df.insert(0, 'user_id', range(1, len(df) + 1))


모순 1: 0건
모순 2: 0건
모순 3: 0건
모순 4: 2건
모순 5: 0건
모순 6: 1건
총 제거: 3건


In [15]:
users = (df['uses_platform'] == '예').sum()
non_users = (df['uses_platform'] == '아니오').sum()

print(f'패션 플랫폼 사용자: {users}명')
print(f'패션 플랫폼 비사용자: {non_users}명')

패션 플랫폼 사용자: 200명
패션 플랫폼 비사용자: 66명


---

## 최종 검증
MySQL 적재 전 shape, 결측값, 타입을 한 번에 점검한다.

In [16]:
print(f'최종 shape: {df.shape}')
print(f'\n결측값:\n{df.isnull().sum()[df.isnull().sum() > 0]}')
print(f'\n타입:\n{df.dtypes}')

최종 shape: (266, 20)

결측값:
platforms             66
selection_factors     66
open_purpose          66
purchase_count        66
last_purchase         75
avg_spend             75
repurchase_reason     75
dissatisfaction       66
continue_use          66
nps                   66
discovery             66
influence             66
feedback             173
dtype: int64

타입:
user_id                       int64
timestamp            datetime64[ns]
gender                       object
age                          object
content_freq               category
monthly_spend              category
uses_platform                object
platforms                    object
selection_factors            object
open_purpose                 object
purchase_count             category
last_purchase              category
avg_spend                  category
repurchase_reason            object
dissatisfaction              object
continue_use               category
nps                           Int64
discovery          

---

## MySQL 적재
정제된 데이터를 `survey` 테이블에 저장하고 행 수로 적재 결과를 검증한다.

In [17]:
load_dotenv()

DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_HOST = os.getenv('DB_HOST')
DB_NAME = os.getenv('DB_NAME')

# DB가 없으면 생성하고, 기존 DB는 유지한다.
engine_root = create_engine(f"mysql+mysqlconnector://{DB_USER}:{DB_PASSWORD}@{DB_HOST}")
with engine_root.connect() as conn:
    conn.execute(text(f"CREATE DATABASE IF NOT EXISTS `{DB_NAME}` CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci"))

engine = create_engine(f"mysql+mysqlconnector://{DB_USER}:{DB_PASSWORD}@{DB_HOST}/{DB_NAME}")
print(f"{DB_HOST} / {DB_NAME} 준비 완료")

localhost / fashion_platform 준비 완료


In [18]:
df.to_sql('survey', con=engine, if_exists='replace', index=False)

# user_id를 PRIMARY KEY로 지정 (조회·JOIN 효율 ↑)
with engine.connect() as conn:
    conn.execute(text("ALTER TABLE survey MODIFY user_id INT NOT NULL PRIMARY KEY"))

cnt = pd.read_sql('SELECT COUNT(*) as n FROM survey', engine).iloc[0, 0]
print(f'survey: {cnt:,}행 적재 완료 (PK: user_id)')

survey: 266행 적재 완료 (PK: user_id)


---

## 전처리 결과 요약

| 항목 | 결과 |
|---|---|
| 원본 응답 | 269명 |
| 제거 응답 | 논리 모순 3건 |
| 최종 분석 기준 | 266명 |
| 플랫폼 사용자 | 200명 |
| 최근 6개월 내 구매자 | 191명 |
| 개인정보 처리 | 이메일 컬럼 분석 데이터셋에서 제외 |
| 유입 경로 처리 | Q16 자유입력성 응답 정규화 |
| 저장 테이블 | MySQL `fashion_platform.survey` |

이후 노트북은 위 모수를 기준으로 분석한다. 특히 `전체 응답자 266명`, `플랫폼 사용자 200명`, `구매자 191명`은 분석 질문에 따라 분모가 달라지므로 각 결과 해석에서 구분해 사용한다.
